# 语言模型：自动补全

在本作业中，你将构建一个自动补全系统。自动补全系统是你每天可能都会见到的东西：
- 当你使用谷歌搜索时，经常会看到帮助你完成搜索的建议。
- 当你撰写电子邮件时，你会收到建议，告诉你句子可能的结尾。

在本作业结束时，你将开发出这样一个系统的原型。

<img src = "stanford.png" style="width:700px;height:300px;"/>

## 目录
- [1 加载和预处理数据](#1)
- [1.1：加载数据](#1.1)
- [1.2 预处理数据](#1.2)
    - [练习 01](#ex-01)
    - [练习 02](#ex-02)
    - [练习 03](#ex-03)
    - [练习 04](#ex-04)
    - [练习 05](#ex-05)
    - [练习 06](#ex-06)
    - [练习 07](#ex-07)
- [2 开发基于 n-gram 的语言模型](#2)
    - [练习 08](#ex-08)
    - [练习 09](#ex-09)    
- [3 困惑度](#3)
    - [练习 10](#ex-10)
- [4 构建自动补全系统](#4)
    - [练习 11](#ex-11)

自动补全系统的关键组成部分是语言模型。
语言模型为单词序列分配概率，使“更可能”的序列获得更高的分数。例如，
>“I have a pen”
预计会比
>“I am a pen”
获得更高的概率，因为前者在现实世界中看起来更像一个自然的句子。

你可以利用这种概率计算来开发自动补全系统。
假设用户输入了
>“I eat scrambled”
那么你可以找到一个单词 `x`，使得“I eat scrambled x”获得最高概率。如果 x = “eggs”，则句子为
>“I eat scrambled eggs”

虽然已经开发了多种语言模型，但本作业使用 **N-grams**，这是一种简单但强大的语言建模方法。
- N-gram 也用于机器翻译和语音识别。

本作业的步骤如下：

1. 加载和预处理数据
    - 加载并标记化数据。
    - 将句子拆分为训练集和测试集。
    - 将低频词替换为未知标记 `<unk>`。
1. 开发基于 N-gram 的语言模型
    - 从给定数据集中计算 n-gram 的计数。
    - 使用 k 平滑估计下一个单词的条件概率。
1. 通过计算困惑度分数来评估 N-gram 模型。
1. 使用你自己的模型，根据给定句子建议下一个单词。

In [1]:
import math
import random
import numpy as np
import pandas as pd
import nltk
nltk.data.path.append('.')
# NLTK 3.9+ requires this resource for nltk.word_tokenize.
nltk.download('punkt_tab', quiet=True)

True

<a name='1'></a>
## 第 1 部分：加载和预处理数据

<a name='1.1'></a>
### 第 1.1 部分：加载数据
你将使用推特数据。
运行下一个单元格来加载数据并查看前几个句子。

请注意，数据是一个长字符串，包含许多推文。
观察到推文之间有换行符 "\n"。

In [2]:
with open("en_US.twitter.txt", "r") as f:
    data = f.read()
print("Data type:", type(data))
print("Number of letters:", len(data))
print("First 300 letters of the data")
print("-------")
display(data[0:300])
print("-------")

print("Last 300 letters of the data")
print("-------")
display(data[-300:])
print("-------")

Data type: <class 'str'>
Number of letters: 3335477
First 300 letters of the data
-------


"How are you? Btw thanks for the RT. You gonna be in DC anytime soon? Love to see you. Been way, way too long.\nWhen you meet someone special... you'll know. Your heart will beat more rapidly and you'll smile for no reason.\nthey've decided its more fun if I don't.\nSo Tired D; Played Lazer Tag & Ran A "

-------
Last 300 letters of the data
-------


"ust had one a few weeks back....hopefully we will be back soon! wish you the best yo\nColombia is with an 'o'...“: We now ship to 4 countries in South America (fist pump). Please welcome Columbia to the Stunner Family”\n#GutsiestMovesYouCanMake Giving a cat a bath.\nCoffee after 5 was a TERRIBLE idea.\n"

-------


<a name='1.2'></a>
### 第 1.2 部分：预处理数据

按照以下步骤预处理此数据：

1. 使用 "\n" 作为分隔符将数据拆分为句子。
1. 将每个句子拆分为标记。请注意，在本作业中，“标记”和“单词”可以互换使用。
1. 将句子分配到训练集或测试集中。
1. 找出在训练数据中至少出现 N 次的标记。
1. 将出现次数少于 N 次的标记替换为 `<unk>`


注意：本练习中省略了验证数据。
- 在实际应用中，我们应保留一部分数据作为验证集，并用它来调整我们的训练。
- 为简便起见，我们跳过此过程。

<a name='ex-01'></a>
### Exercise 01

Split data into sentences.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li> Use <a href="https://docs.python.org/3/library/stdtypes.html?highlight=split#str.split" >str.split</a> </li>
</ul>
</p>

In [3]:
# UNQ_C1 (唯一单元格标识符，请勿编辑)
### 计分函数：split_to_sentences ###
def split_to_sentences(data):
    """
    按换行符 "\n" 拆分数据
    
    参数：
        data: 字符串
    
    返回：
        句子列表
    """
    ### 在此处开始编写代码（将 'None' 替换为你的代码）###
    sentences = data.split("\n")
    ### 在此处结束代码 ###
    
    # 额外的清理（此部分已实现）
    # - 去除每个句子开头和结尾的空格
    # - 如果句子为空字符串，则丢弃
    sentences = [s.strip() for s in sentences]
    sentences = [s for s in sentences if len(s) > 0]
    
    return sentences

In [4]:
# test your code
x = """
I have a pen.\nI have an apple. \nAh\nApple pen.\n
"""
print(x)

split_to_sentences(x)


I have a pen.
I have an apple. 
Ah
Apple pen.




['I have a pen.', 'I have an apple.', 'Ah', 'Apple pen.']

Expected answer: 
```CPP
['I have a pen.', 'I have an apple.', 'Ah', 'Apple pen.']
```

<a name='ex-02'></a>
### 练习 02
下一步是对句子进行标记化（将句子拆分为单词列表）。
- 将所有标记转换为小写，以便原始文本中大写（例如在句首）的单词与小写版本的单词被视为相同。
- 将每个标记化后的单词列表追加到标记化句子的列表中。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>Use <a href="https://docs.python.org/3/library/stdtypes.html?highlight=split#str.lower" >str.lower</a> to convert strings to lowercase. </li>
    <li>Please use <a href="https://www.nltk.org/api/nltk.tokenize.html#nltk.tokenize.punkt.PunktLanguageVars.word_tokenize" >nltk.word_tokenize</a> to split sentences into tokens.</li>
    <li>If you used str.split insteaad of nltk.word_tokenize, there are additional edge cases to handle, such as the punctuation (comma, period) that follows a word.</li>
</ul>
</p>


In [5]:
# UNQ_C2 (唯一单元格标识符，请勿编辑)
### 计分函数：tokenize_sentences ###
def tokenize_sentences(sentences):
    """
    将句子标记化为标记（单词）
    
    参数：
        sentences: 字符串列表
    
    返回：
        标记化句子的列表（每个句子为一个单词列表）
    """
    
    # 初始化标记化句子的列表
    tokenized_sentences = []
    ### 在此处开始编写代码（将 'None' 替换为你的代码）###
    
    # 遍历每个句子
    for sentence in sentences:
        
        # 转换为小写字母
        sentence = sentence.lower()
        
        # 转换为单词列表
        tokenized = nltk.word_tokenize(sentence)
        
        # 将单词列表追加到列表的列表
        tokenized_sentences.append(tokenized)
    
    ### 在此处结束代码 ###
    
    return tokenized_sentences

In [6]:
# test your code
sentences = ["Sky is blue.", "Leaves are green.", "Roses are red."]
tokenize_sentences(sentences)

[['sky', 'is', 'blue', '.'],
 ['leaves', 'are', 'green', '.'],
 ['roses', 'are', 'red', '.']]

### Expected output

```CPP
[['sky', 'is', 'blue', '.'],
 ['leaves', 'are', 'green', '.'],
 ['roses', 'are', 'red', '.']]
```

<a name='ex-03'></a>
### 练习 03

使用你刚刚实现的两个函数来获取标记化后的数据。
- 将数据拆分为句子
- 对这些句子进行标记化

In [7]:
# UNQ_C3 (唯一单元格标识符，请勿编辑)
### 计分函数：get_tokenized_data ###
def get_tokenized_data(data):
    """
    生成标记化句子的列表
    
    参数：
        data: 字符串
    
    返回：
        标记化句子的列表（每个句子为一个单词列表）
    """
    ### 在此处开始编写代码（将 'None' 替换为你的代码）###
    
    # 通过拆分数据获取句子
    sentences = split_to_sentences(data)
    
    # 通过对句子进行标记化来获取标记化句子的列表
    tokenized_sentences = tokenize_sentences(sentences)
    
    ### 在此处结束代码 ###
    
    return tokenized_sentences

In [8]:
# test your function
x = "Sky is blue.\nLeaves are green\nRoses are red."
get_tokenized_data(x)

[['sky', 'is', 'blue', '.'],
 ['leaves', 'are', 'green'],
 ['roses', 'are', 'red', '.']]

##### Expected outcome

```CPP
[['sky', 'is', 'blue', '.'],
 ['leaves', 'are', 'green'],
 ['roses', 'are', 'red', '.']]
```

### Split into train and test sets

Now run the cell below to split data into training and test sets.

In [9]:
tokenized_data = get_tokenized_data(data)
random.seed(87)
random.shuffle(tokenized_data)

train_size = int(len(tokenized_data) * 0.8)
train_data = tokenized_data[0:train_size]
test_data = tokenized_data[train_size:]

In [10]:
print("{} data are split into {} train and {} test set".format(
    len(tokenized_data), len(train_data), len(test_data)))

print("First training sample:")
print(train_data[0])
      
print("First test sample")
print(test_data[0])

47961 data are split into 38368 train and 9593 test set
First training sample:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the', 'team', 'local', 'company', 'and', 'quality', 'production']
First test sample
['that', 'picture', 'i', 'just', 'seen', 'whoa', 'dere', '!', '!', '>', '>', '>', '>', '>', '>', '>']


##### Expected output

```CPP
47961 data are split into 38368 train and 9593 test set
First training sample:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the', 'team', 'local', 'company', 'and', 'quality', 'production']
First test sample
['that', 'picture', 'i', 'just', 'seen', 'whoa', 'dere', '!', '!', '>', '>', '>', '>', '>', '>', '>']
```

<a name='ex-04'></a>
### 练习 04

你不会使用数据中出现的所有标记（单词）进行训练。相反，你将使用更频繁出现的词。
- 你将关注数据中至少出现 N 次的词。
- 首先统计每个单词在数据中出现的次数。

你将需要一个双重 for 循环，一个用于遍历句子，另一个用于遍历句子中的标记。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>If you decide to import and use defaultdict, remember to cast the dictionary back to a regular 'dict' before returning it. </li>
</ul>
</p>


In [11]:
# UNQ_C4 (唯一单元格标识符，请勿编辑)
### 计分函数：count_words ###
def count_words(tokenized_sentences):
    """
    统计标记化句子中单词出现的次数
    
    参数：
        tokenized_sentences: 字符串列表的列表
    
    返回：
        将单词（str）映射到其频率（int）的字典
    """
        
    word_counts = {}
    ### 在此处开始编写代码（将 'None' 替换为你的代码）###
    
    # 遍历每个句子
    for sentence in tokenized_sentences: # 补全此行
        
        # 遍历句子中的每个标记
        for token in sentence: # 补全此行

            # 如果标记尚未在字典中，则将其计数设置为 1
            if token not in word_counts: # 补全此行
                word_counts[token] = 1
            # 如果标记已在字典中，则将其计数加 1
            else:
                word_counts[token] += 1

    ### 在此处结束代码 ###
    
    return word_counts

In [12]:
# test your code
tokenized_sentences = [['sky', 'is', 'blue', '.'],
                       ['leaves', 'are', 'green', '.'],
                       ['roses', 'are', 'red', '.']]
count_words(tokenized_sentences)

{'sky': 1,
 'is': 1,
 'blue': 1,
 '.': 3,
 'leaves': 1,
 'are': 2,
 'green': 1,
 'roses': 1,
 'red': 1}

##### Expected output

Note that the order may differ.

```CPP
{'sky': 1,
 'is': 1,
 'blue': 1,
 '.': 3,
 'leaves': 1,
 'are': 2,
 'green': 1,
 'roses': 1,
 'red': 1}
```

### 处理“词汇表外”单词

如果你的模型正在执行自动补全，但遇到了一个在训练期间从未见过的单词，它将没有输入词来帮助确定下一个要建议的单词。该模型将无法预测下一个单词，因为当前单词没有计数。
- 这种“新”词被称为“未知词”，或称为 **词汇表外（OOV）** 单词。
- 测试集中未知词所占的百分比称为 **OOV 率**。

为了在预测期间处理未知词，请使用一个特殊标记来表示所有未知词 'unk'。
- 修改训练数据，使其包含一些用于训练的“未知”词。
- 要转换为“未知”词的单词是在训练集中出现频率不高的词。
- 创建一个训练集中最频繁单词的列表，称为 **封闭词汇表**。
- 将不属于封闭词汇表的所有其他单词转换为标记 'unk'。

<a name='ex-05'></a>
### 练习 05

现在你将创建一个函数，该函数接受一个文本文档和一个阈值 `count_threshold`。
- 任何计数大于或等于阈值 `count_threshold` 的单词都会保留在封闭词汇表中。
- 返回封闭词汇表列表。

In [13]:
# UNQ_C5 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
### GRADED_FUNCTION: get_words_with_nplus_frequency ###
def get_words_with_nplus_frequency(tokenized_sentences, count_threshold):
    """
    Find the words that appear N times or more
    
    Args:
        tokenized_sentences: List of lists of sentences
        count_threshold: minimum number of occurrences for a word to be in the closed vocabulary.
    
    Returns:
        List of words that appear N times or more
    """
    # Initialize an empty list to contain the words that
    # appear at least 'minimum_freq' times.
    closed_vocab = []
    
    # Get the word couts of the tokenized sentences
    # Use the function that you defined earlier to count the words
    word_counts = count_words(tokenized_sentences)
    
    ### START CODE HERE (Replace instances of 'None' with your code) ###

    # for each word and its count
    for word, cnt in word_counts.items(): # complete this line
        
        # check that the word's count
        # is at least as great as the minimum count
        if cnt >= count_threshold:
            
            # append the word to the list
            closed_vocab.append(word)
    ### END CODE HERE ###
    
    return closed_vocab

In [14]:
# test your code
tokenized_sentences = [['sky', 'is', 'blue', '.'],
                       ['leaves', 'are', 'green', '.'],
                       ['roses', 'are', 'red', '.']]
tmp_closed_vocab = get_words_with_nplus_frequency(tokenized_sentences, count_threshold=2)
print(f"Closed vocabulary:")
print(tmp_closed_vocab)

Closed vocabulary:
['.', 'are']


##### Expected output

```CPP
Closed vocabulary:
['.', 'are']
```

<a name='ex-06'></a>
### 练习 06

出现次数大于或等于 `count_threshold` 次的单词属于封闭词汇表。
- 所有其他单词均被视为 `unknown`。
- 将不在封闭词汇表中的单词替换为标记 `<unk>`。

In [15]:
# UNQ_C6 (唯一单元格标识符，请勿编辑)
### 计分函数：replace_oov_words_by_unk ###
def replace_oov_words_by_unk(tokenized_sentences, vocabulary, unknown_token="<unk>"):
    """
    将不在给定词汇表中的单词替换为 '<unk>' 标记。
    
    参数：
        tokenized_sentences: 字符串列表的列表
        vocabulary: 我们将使用的字符串列表
        unknown_token: 表示未知（词汇表外）单词的字符串
    
    返回：
        字符串列表的列表，其中不在词汇表中的单词已被替换
    """
    
    # 将词汇表放入集合中以加快搜索速度
    vocabulary = set(vocabulary)
    
    # 初始化一个列表，用于存放
    # 低频词被未知标记替换后的句子
    replaced_tokenized_sentences = []
    
    # 遍历每个句子
    for sentence in tokenized_sentences:
        
        # 初始化一个列表，用于存放
        # 包含 "unknown_token" 替换后的单个句子
        replaced_sentence = []
        ### 在此处开始编写代码（将 'None' 替换为你的代码）###

        # 遍历句子中的每个标记
        for token in sentence: # 补全此行
            
            # 检查该标记是否在封闭词汇表中
            if token in vocabulary: # 补全此行
                # 如果是，则将单词追加到 replaced_sentence
                replaced_sentence.append(token)
            else:
                # 否则，改为追加未知标记
                replaced_sentence.append(unknown_token)
        ### 在此处结束代码 ###
        
        # 将标记列表追加到列表的列表
        replaced_tokenized_sentences.append(replaced_sentence)
        
    return replaced_tokenized_sentences

In [16]:
tokenized_sentences = [["dogs", "run"], ["cats", "sleep"]]
vocabulary = ["dogs", "sleep"]
tmp_replaced_tokenized_sentences = replace_oov_words_by_unk(tokenized_sentences, vocabulary)
print(f"Original sentence:")
print(tokenized_sentences)
print(f"tokenized_sentences with less frequent words converted to '<unk>':")
print(tmp_replaced_tokenized_sentences)

Original sentence:
[['dogs', 'run'], ['cats', 'sleep']]
tokenized_sentences with less frequent words converted to '<unk>':
[['dogs', '<unk>'], ['<unk>', 'sleep']]


### Expected answer

```CPP
Original sentence:
[['dogs', 'run'], ['cats', 'sleep']]
tokenized_sentences with less frequent words converted to '<unk>':
[['dogs', '<unk>'], ['<unk>', 'sleep']]
```

<a name='ex-07'></a>
### 练习 07

现在我们已经准备好通过组合你刚刚实现的函数来处理数据。

1. 找出在训练数据中至少出现 count_threshold 次的标记。
1. 对于训练数据和测试数据，都将出现次数少于 count_threshold 次的标记替换为 "<unk\>"。

In [17]:
# UNQ_C7 (唯一单元格标识符，请勿编辑)
### 计分函数：preprocess_data ###
def preprocess_data(train_data, test_data, count_threshold):
    """
    预处理数据，即：
        - 找出在训练数据中至少出现 N 次的标记。
        - 对于训练数据和测试数据，都将出现次数少于 N 次的标记替换为 "<unk>"。        
    参数：
        train_data, test_data: 字符串列表的列表。
        count_threshold: 计数低于此值的单词将被视为未知词。
    
    返回：
        包含以下内容的元组：
        - 将低频词替换为 "<unk>" 后的训练数据
        - 将低频词替换为 "<unk>" 后的测试数据
        - 在训练数据中出现 n 次或以上的词汇表
    """
    ### 在此处开始编写代码（将 'None' 替换为你的代码）###

    # 使用训练数据获取封闭词汇表
    vocabulary = get_words_with_nplus_frequency(train_data, count_threshold)
    
    # 对于训练数据，将不太常见的单词替换为 "<unk>"
    train_data_replaced = replace_oov_words_by_unk(train_data, vocabulary)
    
    # 对于测试数据，将不太常见的单词替换为 "<unk>"
    test_data_replaced = replace_oov_words_by_unk(test_data, vocabulary)
    
    ### 在此处结束代码 ###
    return train_data_replaced, test_data_replaced, vocabulary

In [18]:
# test your code
tmp_train = [['sky', 'is', 'blue', '.'],
     ['leaves', 'are', 'green']]
tmp_test = [['roses', 'are', 'red', '.']]

tmp_train_repl, tmp_test_repl, tmp_vocab = preprocess_data(tmp_train, 
                                                           tmp_test, 
                                                           count_threshold = 1)

print("tmp_train_repl")
print(tmp_train_repl)
print()
print("tmp_test_repl")
print(tmp_test_repl)
print()
print("tmp_vocab")
print(tmp_vocab)

tmp_train_repl
[['sky', 'is', 'blue', '.'], ['leaves', 'are', 'green']]

tmp_test_repl
[['<unk>', 'are', '<unk>', '.']]

tmp_vocab
['sky', 'is', 'blue', '.', 'leaves', 'are', 'green']


##### Expected outcome

```CPP
tmp_train_repl
[['sky', 'is', 'blue', '.'], ['leaves', 'are', 'green']]

tmp_test_repl
[['<unk>', 'are', '<unk>', '.']]

tmp_vocab
['sky', 'is', 'blue', '.', 'leaves', 'are', 'green']
```

### Preprocess the train and test data
Run the cell below to complete the preprocessing both for training and test sets.

In [19]:
minimum_freq = 2
train_data_processed, test_data_processed, vocabulary = preprocess_data(train_data, 
                                                                        test_data, 
                                                                        minimum_freq)

In [20]:
print("First preprocessed training sample:")
print(train_data_processed[0])
print()
print("First preprocessed test sample:")
print(test_data_processed[0])
print()
print("First 10 vocabulary:")
print(vocabulary[0:10])
print()
print("Size of vocabulary:", len(vocabulary))

First preprocessed training sample:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the', 'team', 'local', 'company', 'and', 'quality', 'production']

First preprocessed test sample:
['that', 'picture', 'i', 'just', 'seen', 'whoa', 'dere', '!', '!', '>', '>', '>', '>', '>', '>', '>']

First 10 vocabulary:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the']

Size of vocabulary: 14814


##### Expected output

```CPP
First preprocessed training sample:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the', 'team', 'local', 'company', 'and', 'quality', 'production']

First preprocessed test sample:
['that', 'picture', 'i', 'just', 'seen', 'whoa', 'dere', '!', '!', '>', '>', '>', '>', '>', '>', '>']

First 10 vocabulary:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the']

Size of vocabulary: 14821
```

You are done with the preprocessing section of the assignment.
Objects `train_data_processed`, `test_data_processed`, and `vocabulary` will be used in the rest of the exercises.

<a name='2'></a>
## 第 2 部分：开发基于 n-gram 的语言模型

在本节中，你将开发 n-gram 语言模型。
- 假设下一个词的概率仅取决于前一个 n-gram。
- 前一个 n-gram 是前 'n' 个词的序列。

句子中位置 't' 处的单词，在给定其前面的词为 $w_{t-1}, w_{t-2} \cdots w_{t-n}$ 的条件下的条件概率为：

$$ P(w_t | w_{t-1}\dots w_{t-n}) \tag{1}$$

你可以通过统计这些词序列在训练数据中出现的次数来估计该概率。
- 该概率可以估计为一个比值，其中
- 分子是在训练数据中，词 t-1 到 t-n 出现后，词 't' 紧随其后出现的次数。
- 分母是词 t-1 到 t-n 在训练数据中出现的次数。

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n)}{C(w_{t-1}\dots w_{t-n})} \tag{2} $$

- 函数 $C(\cdots)$ 表示给定序列的出现次数。
- $\hat{P}$ 表示对 $P$ 的估计。
- 注意方程 (2) 的分母是前 $n$ 个词的出现次数，而分子是相同的序列后接单词 $w_t$。

稍后，你将通过添加 k 平滑来修改方程 (2)，以避免任何计数为零时出现错误。

方程 (2) 告诉我们，要基于 n-gram 估计概率，你需要 n-gram 的计数（用于分母）和 (n+1)-gram 的计数（用于分子）。

<a name='ex-08'></a>
### 练习 08
接下来，你将实现一个函数，用于计算任意数量 $n$ 的 n-gram 计数。

在计算 n-gram 的计数时，请事先在句子前添加 $n-1$ 个起始标记 "<s\>" 以表示句子的开始。
- 例如，在二元组模型（N=2）中，包含两个起始标记 "<s\><s\>" 的序列应预测句子的第一个词。
- 因此，如果句子是 "I like food"，则将其修改为 "<s\><s\> I like food"。
- 同时，通过在句尾附加结束标记 "<e\>" 来为计数准备句子，以便模型可以预测何时结束句子。

技术说明：在本实现中，你将把计数存储为字典。
- 字典中每个键值对的键是一个由 n 个词组成的 **元组**（而不是列表）。
- 键值对中的值是出现次数。
- 使用元组作为键而不是列表的原因是，Python 中的列表是可变对象（在首次创建后可以更改）。而元组是“不可变的”，因此在首次创建后无法更改。这使得元组适合作为字典中键的数据类型。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li> To prepend or append, you can create lists and concatenate them using the + operator </li>
    <li> To create a list of a repeated value, you can follow this syntax: <code>['a'] * 3</code> to get <code>['a','a','a']</code> </li>
    <li>To set the range for index 'i', think of this example: An n-gram where n=2 (bigram), and the sentence is length N=5 (including two start tokens and one end token).  So the index positions are <code>[0,1,2,3,4]</code>.  The largest index 'i' where a bigram can start is at position i=3, because the word tokens at position 3 and 4 will form the bigram. </li>
    <li>Remember that the <code>range()</code> function excludes the value that is used for the maximum of the range.  <code> range(3) </code> produces (0,1,2) but excludes 3. </li>
</ul>
</p>


In [ ]:
# UNQ_C8 (唯一单元格标识符，请勿编辑)
### 计分函数：count_n_grams ###
def count_n_grams(data, n, start_token='<s>', end_token = '<e>'):
    """
    统计数据中所有 n-gram 的计数
    
    参数：
        data: 单词列表的列表
        n: 序列中的单词数量
    
    返回：
        将 n 个单词的元组映射到其频率的字典
    """
    
    # 初始化 n-gram 及其计数的字典
    n_grams = {}

    ### 在此处开始编写代码（将 'None' 替换为你的代码）###
    
    # 遍历数据中的每个句子
    for sentence in None: # 补全此行
        #print(sentence)
        # 在句子前添加 n 次起始标记，并在末尾添加 <e> 一次
        sentence = [start_token] * n + sentence + [end_token]
        
        # 将列表转换为元组
        # 以便词序列可以用作字典中的键
        sentence = tuple(sentence)
        
        # 使用 'i' 表示 n-gram 的起始位置
        # 从索引 0 开始
        # 到 n-gram 结尾仍在句子范围内的最后一个索引
        
        for i in None: # 补全此行

            # 获取从 i 到 i+n 的 n-gram
            n_gram = None

            # 检查该 n-gram 是否在字典中
            if None: # 补全此行
            
                # 增加此 n-gram 的计数
                n_grams[None] += None
            else:
                # 将此 n-gram 的计数初始化为 1
                n_grams[None] = None
    
            ### 在此处结束代码 ###
    return n_grams

In [ ]:
# test your code
# CODE REVIEW COMMENT: Outcome does not match expected outcome
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
print("Uni-gram:")
print(count_n_grams(sentences, 1))
print("Bi-gram:")
print(count_n_grams(sentences, 2))

Expected outcome:

```CPP
Uni-gram:
{('<s>',): 2, ('i',): 1, ('like',): 2, ('a',): 2, ('cat',): 2, ('<e>',): 2, ('this',): 1, ('dog',): 1, ('is',): 1}
Bi-gram:
{('<s>', '<s>'): 2, ('<s>', 'i'): 1, ('i', 'like'): 1, ('like', 'a'): 2, ('a', 'cat'): 2, ('cat', '<e>'): 2, ('<s>', 'this'): 1, ('this', 'dog'): 1, ('dog', 'is'): 1, ('is', 'like'): 1}
```

<a name='ex-09'></a>
### Exercise 09

Next, estimate the probability of a word given the prior 'n' words using the n-gram counts.

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n)}{C(w_{t-1}\dots w_{t-n})} \tag{2} $$

This formula doesn't work when a count of an n-gram is zero..
- Suppose we encounter an n-gram that did not occur in the training data.  
- Then, the equation (2) cannot be evaluated (it becomes zero divided by zero).

A way to handle zero counts is to add k-smoothing.  
- K-smoothing adds a positive constant $k$ to each numerator and $k \times |V|$ in the denominator, where $|V|$ is the number of words in the vocabulary.

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n) + k}{C(w_{t-1}\dots w_{t-n}) + k|V|} \tag{3} $$


For n-grams that have a zero count, the equation (3) becomes $\frac{1}{|V|}$.
- This means that any n-gram with zero count has the same probability of $\frac{1}{|V|}$.

Define a function that computes the probability estimate (3) from n-gram counts and a constant $k$.

- The function takes in a dictionary 'n_gram_counts', where the key is the n-gram and the value is the count of that n-gram.
- The function also takes another dictionary n_plus1_gram_counts, which you'll use to find the count for the previous n-gram plus the current word.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>To define a tuple containing a single value, add a comma after that value.  For example: <code>('apple',)</code> is a tuple containing a single string 'apple' </li>
    <li>To concatenate two tuples, use the '+' operator</li>
    <li><a href="" > words </a> </li>
</ul>
</p>


In [ ]:
# UNQ_C9 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
### GRADED FUNCTION: estimate_probability ###
def estimate_probability(word, previous_n_gram, 
                         n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):
    """
    Estimate the probabilities of a next word using the n-gram counts with k-smoothing
    
    Args:
        word: next word
        previous_n_gram: A sequence of words of length n
        n_gram_counts: Dictionary of counts of n-grams
        n_plus1_gram_counts: Dictionary of counts of (n+1)-grams
        vocabulary_size: number of words in the vocabulary
        k: positive constant, smoothing parameter
    
    Returns:
        A probability
    """
    # convert list to tuple to use it as a dictionary key
    previous_n_gram = tuple(previous_n_gram)
    
    ### START CODE HERE (Replace instances of 'None' with your code) ###
    
    # Set the denominator
    # If the previous n-gram exists in the dictionary of n-gram counts,
    # Get its count.  Otherwise set the count to zero
    # Use the dictionary that has counts for n-grams
    previous_n_gram_count = None
        
    # Calculate the denominator using the count of the previous n gram
    # and apply k-smoothing
    denominator = None

    # Define n plus 1 gram as the previous n-gram plus the current word as a tuple
    n_plus1_gram = None

    # Set the count to the count in the dictionary,
    # otherwise 0 if not in the dictionary
    # use the dictionary that has counts for the n-gram plus current word
    n_plus1_gram_count = None

    # Define the numerator use the count of the n-gram plus current word,
    # and apply smoothing
    numerator = None

    # Calculate the probability as the numerator divided by denominator
    probability = None

    ### END CODE HERE ###
    
    return probability

In [ ]:
# test your code
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)

tmp_prob = estimate_probability("cat", "a", unigram_counts, bigram_counts, len(unique_words), k=1)

print(f"The estimated probability of word 'cat' given the previous n-gram 'a' is: {tmp_prob:.4f}")

##### Expected output

```CPP
The estimated probability of word 'cat' given the previous n-gram 'a' is: 0.3333
```

### Estimate probabilities for all words

The function defined below loops over all words in vocabulary to calculate probabilities for all possible words.
- This function is provided for you.

In [ ]:
def estimate_probabilities(previous_n_gram, n_gram_counts, n_plus1_gram_counts, vocabulary, k=1.0):
    """
    Estimate the probabilities of next words using the n-gram counts with k-smoothing
    
    Args:
        previous_n_gram: A sequence of words of length n
        n_gram_counts: Dictionary of counts of (n+1)-grams
        n_plus1_gram_counts: Dictionary of counts of (n+1)-grams
        vocabulary: List of words
        k: positive constant, smoothing parameter
    
    Returns:
        A dictionary mapping from next words to the probability.
    """
    
    # convert list to tuple to use it as a dictionary key
    previous_n_gram = tuple(previous_n_gram)
    
    # add <e> <unk> to the vocabulary
    # <s> is not needed since it should not appear as the next word
    vocabulary = vocabulary + ["<e>", "<unk>"]
    vocabulary_size = len(vocabulary)
    
    probabilities = {}
    for word in vocabulary:
        probability = estimate_probability(word, previous_n_gram, 
                                           n_gram_counts, n_plus1_gram_counts, 
                                           vocabulary_size, k=k)
        probabilities[word] = probability

    return probabilities

In [ ]:
# test your code
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
estimate_probabilities("a", unigram_counts, bigram_counts, unique_words, k=1)

##### Expected output

```CPP
{'cat': 0.2727272727272727,
 'i': 0.09090909090909091,
 'this': 0.09090909090909091,
 'a': 0.09090909090909091,
 'is': 0.09090909090909091,
 'like': 0.09090909090909091,
 'dog': 0.09090909090909091,
 '<e>': 0.09090909090909091,
 '<unk>': 0.09090909090909091}
```

In [ ]:
# Additional test
trigram_counts = count_n_grams(sentences, 3)
estimate_probabilities(["<s>", "<s>"], bigram_counts, trigram_counts, unique_words, k=1)

##### Expected output

```CPP
{'cat': 0.09090909090909091,
 'i': 0.18181818181818182,
 'this': 0.18181818181818182,
 'a': 0.09090909090909091,
 'is': 0.09090909090909091,
 'like': 0.09090909090909091,
 'dog': 0.09090909090909091,
 '<e>': 0.09090909090909091,
 '<unk>': 0.09090909090909091}
```

### Count and probability matrices

As we have seen so far, the n-gram counts computed above are sufficient for computing the probabilities of the next word.  
- It can be more intuitive to present them as count or probability matrices.
- The functions defined in the next cells return count or probability matrices.
- This function is provided for you.

In [ ]:
def make_count_matrix(n_plus1_gram_counts, vocabulary):
    # add <e> <unk> to the vocabulary
    # <s> is omitted since it should not appear as the next word
    vocabulary = vocabulary + ["<e>", "<unk>"]
    
    # obtain unique n-grams
    n_grams = []
    for n_plus1_gram in n_plus1_gram_counts.keys():
        n_gram = n_plus1_gram[0:-1]
        n_grams.append(n_gram)
    n_grams = list(set(n_grams))
    
    # mapping from n-gram to row
    row_index = {n_gram:i for i, n_gram in enumerate(n_grams)}
    # mapping from next word to column
    col_index = {word:j for j, word in enumerate(vocabulary)}
    
    nrow = len(n_grams)
    ncol = len(vocabulary)
    count_matrix = np.zeros((nrow, ncol))
    for n_plus1_gram, count in n_plus1_gram_counts.items():
        n_gram = n_plus1_gram[0:-1]
        word = n_plus1_gram[-1]
        if word not in vocabulary:
            continue
        i = row_index[n_gram]
        j = col_index[word]
        count_matrix[i, j] = count
    
    count_matrix = pd.DataFrame(count_matrix, index=n_grams, columns=vocabulary)
    return count_matrix

In [ ]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
bigram_counts = count_n_grams(sentences, 2)

print('bigram counts')
display(make_count_matrix(bigram_counts, unique_words))

##### Expected output

```CPP
bigram counts
          cat    i   this   a  is   like  dog  <e>   <unk>
(<s>,)    0.0   1.0  1.0  0.0  0.0  0.0   0.0  0.0    0.0
(a,)      2.0   0.0  0.0  0.0  0.0  0.0   0.0  0.0    0.0
(this,)   0.0   0.0  0.0  0.0  0.0  0.0   1.0  0.0    0.0
(like,)   0.0   0.0  0.0  2.0  0.0  0.0   0.0  0.0    0.0
(dog,)    0.0   0.0  0.0  0.0  1.0  0.0   0.0  0.0    0.0
(cat,)    0.0   0.0  0.0  0.0  0.0  0.0   0.0  2.0    0.0
(is,)     0.0   0.0  0.0  0.0  0.0  1.0   0.0  0.0    0.0
(i,)      0.0   0.0  0.0  0.0  0.0  1.0   0.0  0.0    0.0
```

In [ ]:
# Show trigram counts
print('\ntrigram counts')
trigram_counts = count_n_grams(sentences, 3)
display(make_count_matrix(trigram_counts, unique_words))

##### Expected output

```CPP
trigram counts
              cat    i   this   a  is   like  dog  <e>   <unk>
(dog, is)     0.0   0.0  0.0  0.0  0.0  1.0   0.0  0.0    0.0
(this, dog)   0.0   0.0  0.0  0.0  1.0  0.0   0.0  0.0    0.0
(a, cat)      0.0   0.0  0.0  0.0  0.0  0.0   0.0  2.0    0.0
(like, a)     2.0   0.0  0.0  0.0  0.0  0.0   0.0  0.0    0.0
(is, like)    0.0   0.0  0.0  1.0  0.0  0.0   0.0  0.0    0.0
(<s>, i)      0.0   0.0  0.0  0.0  0.0  1.0   0.0  0.0    0.0
(i, like)     0.0   0.0  0.0  1.0  0.0  0.0   0.0  0.0    0.0
(<s>, <s>)    0.0   1.0  1.0  0.0  0.0  0.0   0.0  0.0    0.0
(<s>, this)   0.0   0.0  0.0  0.0  0.0  0.0   1.0  0.0    0.0
```

The following function calculates the probabilities of each word given the previous n-gram, and stores this in matrix form.
- This function is provided for you.

In [ ]:
def make_probability_matrix(n_plus1_gram_counts, vocabulary, k):
    count_matrix = make_count_matrix(n_plus1_gram_counts, unique_words)
    count_matrix += k
    prob_matrix = count_matrix.div(count_matrix.sum(axis=1), axis=0)
    return prob_matrix

In [ ]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
bigram_counts = count_n_grams(sentences, 2)
print("bigram probabilities")
display(make_probability_matrix(bigram_counts, unique_words, k=1))

In [ ]:
print("trigram probabilities")
trigram_counts = count_n_grams(sentences, 3)
display(make_probability_matrix(trigram_counts, unique_words, k=1))

Confirm that you obtain the same results as for the `estimate_probabilities` function that you implemented.

<a name='3'></a>
## Part 3: Perplexity

In this section, you will generate the perplexity score to evaluate your model on the test set. 
- You will also use back-off when needed. 
- Perplexity is used as an evaluation metric of your language model. 
- To calculate the  the perplexity score of the test set on an n-gram model, use: 

$$ PP(W) =\sqrt[N]{ \prod_{t=n+1}^N \frac{1}{P(w_t | w_{t-n} \cdots w_{t-1})} } \tag{4}$$

- where $N$ is the length of the sentence.
- $n$ is the number of words in the n-gram (e.g. 2 for a bigram).
- In math, the numbering starts at one and not zero.

In code, array indexing starts at zero, so the code will use ranges for $t$ according to this formula:

$$ PP(W) =\sqrt[N]{ \prod_{t=n}^{N-1} \frac{1}{P(w_t | w_{t-n} \cdots w_{t-1})} } \tag{4.1}$$

The higher the probabilities are, the lower the perplexity will be. 
- The more the n-grams tell us about the sentence, the lower the perplexity score will be. 

<a name='ex-10'></a>
### Exercise 10
Compute the perplexity score given an N-gram count matrix and a sentence. 

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>Remember that <code>range(2,4)</code> produces the integers [2, 3] (and excludes 4).</li>
</ul>
</p>


In [ ]:
# UNQ_C10 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: calculate_perplexity
def calculate_perplexity(sentence, n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):
    """
    Calculate perplexity for a list of sentences
    
    Args:
        sentence: List of strings
        n_gram_counts: Dictionary of counts of (n+1)-grams
        n_plus1_gram_counts: Dictionary of counts of (n+1)-grams
        vocabulary_size: number of unique words in the vocabulary
        k: Positive smoothing constant
    
    Returns:
        Perplexity score
    """
    # length of previous words
    n = len(list(n_gram_counts.keys())[0]) 
    
    # prepend <s> and append <e>
    sentence = ["<s>"] * n + sentence + ["<e>"]
    
    # Cast the sentence from a list to a tuple
    sentence = tuple(sentence)
    
    # length of sentence (after adding <s> and <e> tokens)
    N = len(sentence)
    
    # The variable p will hold the product
    # that is calculated inside the n-root
    # Update this in the code below
    product_pi = 1.0
    
    ### START CODE HERE (Replace instances of 'None' with your code) ###
    
    # Index t ranges from n to N - 1, inclusive on both ends
    for t in None: # complete this line

        # get the n-gram preceding the word at position t
        n_gram = None
        
        # get the word at position t
        word = None
        
        # Estimate the probability of the word given the n-gram
        # using the n-gram counts, n-plus1-gram counts,
        # vocabulary size, and smoothing constant
        probability = None
        
        # Update the product of the probabilities
        # This 'product_pi' is a cumulative product 
        # of the (1/P) factors that are calculated in the loop
        product_pi *= None

    # Take the Nth root of the product
    perplexity = None
    
    ### END CODE HERE ### 
    return perplexity

In [ ]:
# test your code

sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)


perplexity_train1 = calculate_perplexity(sentences[0],
                                         unigram_counts, bigram_counts,
                                         len(unique_words), k=1.0)
print(f"Perplexity for first train sample: {perplexity_train1:.4f}")

test_sentence = ['i', 'like', 'a', 'dog']
perplexity_test = calculate_perplexity(test_sentence,
                                       unigram_counts, bigram_counts,
                                       len(unique_words), k=1.0)
print(f"Perplexity for test sample: {perplexity_test:.4f}")

### Expected Output

```CPP
Perplexity for first train sample: 2.8040
Perplexity for test sample: 3.9654
```

<b> Note: </b> If your sentence is really long, there will be underflow when multiplying many fractions.
- To handle longer sentences, modify your implementation to take the sum of the log of the probabilities.

<a name='4'></a>
## Part 4: Build an auto-complete system

In this section, you will combine the language models developed so far to implement an auto-complete system. 


<a name='ex-11'></a>
### Exercise 11
Compute probabilities for all possible next words and suggest the most likely one.
- This function also take an optional argument `start_with`, which specifies the first few letters of the next words.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li><code>estimate_probabilities</code> returns a dictionary where the key is a word and the value is the word's probability.</li>
    <li> Use <code>str1.startswith(str2)</code> to determine if a string starts with the letters of another string.  For example, <code>'learning'.startswith('lea')</code> returns True, whereas <code>'learning'.startswith('ear')</code> returns False. There are two additional parameters in <code>str.startswith()</code>, but you can use the default values for those parameters in this case.</li>
</ul>
</p>

In [ ]:
# UNQ_C11 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: suggest_a_word
def suggest_a_word(previous_tokens, n_gram_counts, n_plus1_gram_counts, vocabulary, k=1.0, start_with=None):
    """
    Get suggestion for the next word
    
    Args:
        previous_tokens: The sentence you input where each token is a word. Must have length > n 
        n_gram_counts: Dictionary of counts of (n+1)-grams
        n_plus1_gram_counts: Dictionary of counts of (n+1)-grams
        vocabulary: List of words
        k: positive constant, smoothing parameter
        start_with: If not None, specifies the first few letters of the next word
        
    Returns:
        A tuple of 
          - string of the most likely next word
          - corresponding probability
    """
    
    # length of previous words
    n = len(list(n_gram_counts.keys())[0]) 
    
    # From the words that the user already typed
    # get the most recent 'n' words as the previous n-gram
    previous_n_gram = previous_tokens[-n:]

    # Estimate the probabilities that each word in the vocabulary
    # is the next word,
    # given the previous n-gram, the dictionary of n-gram counts,
    # the dictionary of n plus 1 gram counts, and the smoothing constant
    probabilities = estimate_probabilities(previous_n_gram,
                                           n_gram_counts, n_plus1_gram_counts,
                                           vocabulary, k=k)
    
    # Initialize suggested word to None
    # This will be set to the word with highest probability
    suggestion = None
    
    # Initialize the highest word probability to 0
    # this will be set to the highest probability 
    # of all words to be suggested
    max_prob = 0
    
    ### START CODE HERE (Replace instances of 'None' with your code) ###
    
    # For each word and its probability in the probabilities dictionary:
    for word, prob in None: # complete this line
        
        # If the optional start_with string is set
        if None: # complete this line
            
            # Check if the beginning of word does not match with the letters in 'start_with'
            if None: # complete this line

                # if they don't match, skip this word (move onto the next word)
                None # complete this line
        
        # Check if this word's probability
        # is greater than the current maximum probability
        if None: # complete this line
            
            # If so, save this word as the best suggestion (so far)
            suggestion = None
            
            # Save the new maximum probability
            max_prob = None

    ### END CODE HERE
    
    return suggestion, max_prob

In [ ]:
# test your code
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)

previous_tokens = ["i", "like"]
tmp_suggest1 = suggest_a_word(previous_tokens, unigram_counts, bigram_counts, unique_words, k=1.0)
print(f"The previous words are 'i like',\n\tand the suggested word is `{tmp_suggest1[0]}` with a probability of {tmp_suggest1[1]:.4f}")

print()
# test your code when setting the starts_with
tmp_starts_with = 'c'
tmp_suggest2 = suggest_a_word(previous_tokens, unigram_counts, bigram_counts, unique_words, k=1.0, start_with=tmp_starts_with)
print(f"The previous words are 'i like', the suggestion must start with `{tmp_starts_with}`\n\tand the suggested word is `{tmp_suggest2[0]}` with a probability of {tmp_suggest2[1]:.4f}")

### Expected output

```CPP
The previous words are 'i like',
	and the suggested word is `a` with a probability of 0.2727

The previous words are 'i like', the suggestion must start with `c`
	and the suggested word is `cat` with a probability of 0.0909

```

### Get multiple suggestions

The function defined below loop over varioud n-gram models to get multiple suggestions.

In [ ]:
def get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0, start_with=None):
    model_counts = len(n_gram_counts_list)
    suggestions = []
    for i in range(model_counts-1):
        n_gram_counts = n_gram_counts_list[i]
        n_plus1_gram_counts = n_gram_counts_list[i+1]
        
        suggestion = suggest_a_word(previous_tokens, n_gram_counts,
                                    n_plus1_gram_counts, vocabulary,
                                    k=k, start_with=start_with)
        suggestions.append(suggestion)
    return suggestions

In [ ]:
# test your code
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
trigram_counts = count_n_grams(sentences, 3)
quadgram_counts = count_n_grams(sentences, 4)
qintgram_counts = count_n_grams(sentences, 5)

n_gram_counts_list = [unigram_counts, bigram_counts, trigram_counts, quadgram_counts, qintgram_counts]
previous_tokens = ["i", "like"]
tmp_suggest3 = get_suggestions(previous_tokens, n_gram_counts_list, unique_words, k=1.0)

print(f"The previous words are 'i like', the suggestions are:")
display(tmp_suggest3)

### Suggest multiple words using n-grams of varying length

Congratulations!  You have developed all building blocks for implementing your own auto-complete systems.

Let's see this with n-grams of varying lengths (unigrams, bigrams, trigrams, 4-grams...6-grams).

In [ ]:
n_gram_counts_list = []
for n in range(1, 6):
    print("Computing n-gram counts with n =", n, "...")
    n_model_counts = count_n_grams(train_data_processed, n)
    n_gram_counts_list.append(n_model_counts)

In [ ]:
previous_tokens = ["i", "am", "to"]
tmp_suggest4 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"The previous words are {previous_tokens}, the suggestions are:")
display(tmp_suggest4)

In [ ]:
previous_tokens = ["i", "want", "to", "go"]
tmp_suggest5 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"The previous words are {previous_tokens}, the suggestions are:")
display(tmp_suggest5)

In [ ]:
previous_tokens = ["hey", "how", "are"]
tmp_suggest6 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"The previous words are {previous_tokens}, the suggestions are:")
display(tmp_suggest6)

In [ ]:
previous_tokens = ["hey", "how", "are", "you"]
tmp_suggest7 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"The previous words are {previous_tokens}, the suggestions are:")
display(tmp_suggest7)

In [ ]:
previous_tokens = ["hey", "how", "are", "you"]
tmp_suggest8 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0, start_with="d")

print(f"The previous words are {previous_tokens}, the suggestions are:")
display(tmp_suggest8)

# Congratulations!

You've completed this assignment by building an autocomplete model using an n-gram language model!  

Please continue onto the fourth and final week of this course!